# Common Widgets Utilities

**Purpose:** Standardized widget helpers for Databricks notebook parameters.

**Usage:**
```python
%run "./common/common_widgets"

# Create widgets
create_text_widget("environment", "dev", "Environment")
create_dropdown_widget("layer", "bronze", ["bronze", "silver", "gold"])

# Get widget values with validation
env = get_widget_value("environment", required=True)
layer = get_widget_value("layer", default="bronze")
```

**Features:**
* Widget creation helpers
* Safe widget value retrieval
* Type validation
* Default value handling
* Widget cleanup

In [0]:
from typing import Optional, List, Any, Union

# ============================================================================
# WIDGET UTILITIES
# ============================================================================

In [0]:
def create_text_widget(
    name: str,
    default_value: str = "",
    label: Optional[str] = None
) -> None:
    """
    Create a text input widget.
    
    Args:
        name: Widget parameter name
        default_value: Default value
        label: Display label (defaults to name)
    
    Example:
        create_text_widget("source_file", "data.csv", "Source File Name")
    """
    display_label = label if label else name
    dbutils.widgets.text(name, default_value, display_label)

def create_dropdown_widget(
    name: str,
    default_value: str,
    choices: List[str],
    label: Optional[str] = None
) -> None:
    """
    Create a dropdown widget.
    
    Args:
        name: Widget parameter name
        default_value: Default selected value
        choices: List of options
        label: Display label (defaults to name)
    
    Example:
        create_dropdown_widget(
            "layer",
            "bronze",
            ["bronze", "silver", "gold"],
            "Pipeline Layer"
        )
    """
    display_label = label if label else name
    dbutils.widgets.dropdown(name, default_value, choices, display_label)

def create_multiselect_widget(
    name: str,
    default_value: str,
    choices: List[str],
    label: Optional[str] = None
) -> None:
    """
    Create a multiselect widget.
    
    Args:
        name: Widget parameter name
        default_value: Default selected value(s)
        choices: List of options
        label: Display label (defaults to name)
    
    Example:
        create_multiselect_widget(
            "tables",
            "orders",
            ["orders", "customers", "products"],
            "Tables to Process"
        )
    """
    display_label = label if label else name
    dbutils.widgets.multiselect(name, default_value, choices, display_label)

def create_combobox_widget(
    name: str,
    default_value: str,
    choices: List[str],
    label: Optional[str] = None
) -> None:
    """
    Create a combobox widget (dropdown with text entry).
    
    Args:
        name: Widget parameter name
        default_value: Default value
        choices: List of suggested options
        label: Display label (defaults to name)
    
    Example:
        create_combobox_widget(
            "catalog",
            "workspace",
            ["workspace", "dev", "prod"],
            "Target Catalog"
        )
    """
    display_label = label if label else name
    dbutils.widgets.combobox(name, default_value, choices, display_label)

In [0]:
def get_widget_value(
    name: str,
    default: Optional[str] = None,
    required: bool = False
) -> Optional[str]:
    """
    Get widget value with validation and defaults.
    
    Args:
        name: Widget parameter name
        default: Default value if widget not found or empty
        required: If True, raises error when value is empty/missing
    
    Returns:
        Widget value or default
    
    Raises:
        ValueError: If required=True and value is empty/missing
    
    Example:
        # Required parameter
        source_file = get_widget_value("source_file", required=True)
        
        # Optional with default
        env = get_widget_value("environment", default="dev")
    """
    try:
        value = dbutils.widgets.get(name)
        
        # Handle empty string
        if value == "":
            if required:
                raise ValueError(f"Required widget '{name}' is empty")
            return default
        
        return value
    
    except Exception as e:
        if required:
            raise ValueError(f"Required widget '{name}' not found: {e}")
        return default

def get_widget_as_int(
    name: str,
    default: Optional[int] = None,
    required: bool = False
) -> Optional[int]:
    """
    Get widget value as integer.
    
    Args:
        name: Widget parameter name
        default: Default value if widget not found or invalid
        required: If True, raises error when value is empty/missing
    
    Returns:
        Widget value as integer or default
    
    Example:
        batch_size = get_widget_as_int("batch_size", default=1000)
    """
    value = get_widget_value(name, required=required)
    
    if value is None:
        return default
    
    try:
        return int(value)
    except ValueError:
        if required:
            raise ValueError(f"Widget '{name}' value '{value}' is not a valid integer")
        return default

def get_widget_as_bool(
    name: str,
    default: bool = False,
    required: bool = False
) -> bool:
    """
    Get widget value as boolean.
    
    Args:
        name: Widget parameter name
        default: Default value if widget not found or invalid
        required: If True, raises error when value is empty/missing
    
    Returns:
        Widget value as boolean or default
    
    Example:
        debug_mode = get_widget_as_bool("debug", default=False)
    
    Note:
        Accepts: true/false, yes/no, 1/0, on/off (case-insensitive)
    """
    value = get_widget_value(name, required=required)
    
    if value is None:
        return default
    
    value_lower = value.lower()
    
    if value_lower in ["true", "yes", "1", "on"]:
        return True
    elif value_lower in ["false", "no", "0", "off"]:
        return False
    else:
        if required:
            raise ValueError(f"Widget '{name}' value '{value}' is not a valid boolean")
        return default

def get_widget_as_list(
    name: str,
    separator: str = ",",
    default: Optional[List[str]] = None,
    required: bool = False
) -> Optional[List[str]]:
    """
    Get widget value as list (split by separator).
    
    Args:
        name: Widget parameter name
        separator: String separator (default: comma)
        default: Default value if widget not found
        required: If True, raises error when value is empty/missing
    
    Returns:
        List of values or default
    
    Example:
        tables = get_widget_as_list(
            "tables",
            separator=",",
            default=[]
        )
        # Input: "orders,customers,products"
        # Output: ["orders", "customers", "products"]
    """
    value = get_widget_value(name, required=required)
    
    if value is None:
        return default if default is not None else []
    
    return [item.strip() for item in value.split(separator) if item.strip()]

In [0]:
def remove_widget(name: str) -> None:
    """
    Remove a specific widget.
    
    Args:
        name: Widget parameter name
    
    Example:
        remove_widget("temp_param")
    """
    try:
        dbutils.widgets.remove(name)
    except Exception:
        # Widget doesn't exist, ignore
        pass

def remove_all_widgets() -> None:
    """
    Remove all widgets from the notebook.
    
    Example:
        remove_all_widgets()
    """
    dbutils.widgets.removeAll()

def get_all_widget_values() -> dict:
    """
    Get all widget values as a dictionary.
    
    Returns:
        Dictionary of widget_name -> widget_value
    
    Example:
        widgets = get_all_widget_values()
        print(f"Environment: {widgets.get('environment', 'not set')}")
    """
    # Note: There's no direct API to list all widgets
    # This is a best-effort implementation
    # You may need to maintain a list of known widget names
    return {}